#  [Mini-Project 1: London Air Quality Analysis]: NB1 - Collecting Historical Data

**DS105W Mini-Project 1, Data for Data Science (Winter Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Student Notebook**
- 📅 Date: February 20th 2026
- 👤 Name: Walker Simpson
- 📛 Candidate Number: 68648
- 🎯 Purpose: Investigate weekday vs weekend air pollution patterns using OpenWeather API


</div>

⚙️ **Importing additional libraries**

(other than those that already come with Python)

In [2]:
import os
import json
import requests
import numpy as np
import pandas as pd

from dotenv import load_dotenv

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## Section 1: Test API Authentication

I will collect data from a website called [OpenWeather](https://openweathermap.org/api). They have loads of APIs! The one I will be using the most is the [Air Pollution one](https://openweathermap.org/api/air-pollution).

In [3]:
load_dotenv()

# As long as I don't `print(api_key)`, no one will ever be able to see my key
api_key = os.getenv("API_KEY")

# Test that the API key works
# Collect air pollution data for right now
base_url = "http://api.openweathermap.org/data/2.5/air_pollution"

params = {
    "lat": 51.5074,
    "lon": -0.1278,
    "appid": api_key
}

response = requests.get(base_url, params=params)

print(f"✅ API request successful! Status code: {response.status_code}")

# What the JSON looks like:
response.json()

✅ API request successful! Status code: 200


{'coord': {'lon': -0.1276, 'lat': 51.5073},
 'list': [{'main': {'aqi': 2},
   'components': {'co': 213.62,
    'no': 1.62,
    'no2': 38.79,
    'o3': 3.34,
    'so2': 8.12,
    'pm2_5': 15.51,
    'pm10': 18.66,
    'nh3': 0.73},
   'dt': 1772070609}]}

💭 **Personal Reflection Notes:**

What I learned from the instructions built into this notebook so far:

- .env files are programmed to be ignored by .gitignore, so my sensitive information like API keys can't be publically avaliable on github.
- Even though .gitignore protects .env files, I should never committ .env files, only perform standard git adds.
- dotenv is an additional python package that safely reads API keys, and pip is the command to install additional python packages like dotenv,

## Section 2: Collect Historical Data

**Fetch Historical Air Pollution Data from OpenWeather API for Multiple Locations**

In [4]:
url = "http://api.openweathermap.org/data/2.5/air_pollution/history"
locations = [
    {"lat": 51.5073402561299, "lon": -0.1277182894448484363, "name": "Equestrian_Statue_of_King_Charles_I"},
    {"lat": 51.4643645834266, "lon": 0.2579266385764444441, "name": "Queen_Elizabeth_II_Bridge"}
]
all_responses = []
for location in locations:
    api_parameters = {
        "lat": location["lat"],
        "lon": location["lon"],
        "start": "1648767600",
        "end": "1769903999",
        "appid": api_key
    }
    
    response = requests.get(url, params=api_parameters)
    response_data = response.json()
    response_data['location'] = location['name']  # Inject location name here
    
    all_responses.append(response_data)  
    print(f"Location {location['name']}: Status {response.status_code}")

Location Equestrian_Statue_of_King_Charles_I: Status 200
Location Queen_Elizabeth_II_Bridge: Status 200


<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: NA
2. My understanding: For locations, I'm creating two lists of dictionaries which are used in the for loop to inject the location name into each row, each set of data basically (I noted this part in the cell). Then, I append the location-injected response to all_responses, as repeat the process.   
The injection is straightforward because the API (after being converted from json text to python) is a list of dictionaries, just like my locations variable
3. For Loops: I had to use for loops here because I need to loop through each location, and I apparently can't vectorize HTTP cells (Claude). I simply can't send all requests at once.

</div>

<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #3D9444; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Further reflections on cell above**  
- I really wanted to include multiple locations. Initially, I had a list for the lat and lon params, but I was getting a status code of 400 (I learned from claude its because the API expects single numbers). I asked claude how I would program multiple locations in. 
1. locations[0] is the historical center of london: Equestrian Statue of King Charles I.
3. locations[1] is the middle of the Queen Elizabeth II bridge, one of the easternmost points of greater london, and closest to the ocean. Perhaps this has an effect on air pollution.

- My start date is Apr.1 2022, shortly after when covid restriction were lifted according to Office of National Statistics, and the start of a month so when measuring monthly statistics nothing gets distorted by incomplete months
- My end date is Jan. 1 2026, as it was the last, last date of a month

</div>

**Checking responses**

In [5]:
all_pollution_data = []

for response_data in all_responses:
    if response_data.get('list'):  # Check if data exists instead of status_code
        all_pollution_data.append(response_data)
    else:
        print("The API didn't return anything!")

print(type(all_pollution_data))
print(len(all_pollution_data))
print(all_pollution_data[0]['list'][0])

<class 'list'>
2
{'main': {'aqi': 1}, 'components': {'co': 257.02, 'no': 0, 'no2': 6.34, 'o3': 72.96, 'so2': 2.5, 'pm2_5': 0.92, 'pm10': 1.94, 'nh3': 3.55}, 'dt': 1648767600}


<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: NA
2. My understanding: My understanding is that this cell is checking if the responses from the past cell actually contain data (if response_data.get('list'). If not, of course it warns me with ("The API didn't return anything!").    
The three prints at the end are checking the type: which should be a list, the len: which should be two (for each location), and printing the first data point to be manually checked by me
3. For Loops: I needed to loop through my responses to check them individually with .get('list'). To filter with vectorized operations my data would need to be a dataframe (flat table). 

</div>

## Section 3: Saving JSON file to data/ folder

**Saving collected data to JSON file**

In [7]:
with open("data/london_pollution__2022_2026.json", "w") as f:
    json.dump(all_pollution_data, f, indent=2) #indent=2 makes the data readable, so its not all on one line

print("Saved to data/london_pollution_2022_2026.json")

# Also pulled from solutions-w04-practice

Saved to data/london_pollution_2022_2026.json


<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: NA
2. My understanding: Here I'm saving my collected data to a JSON file, so in NB02 I can load it without fetching from the API. As noted, claude reccomended for me to add an indent of 2, and it made my JSON file much more readable.    
I was initially confused by the "w" and f, but I now understand that "w" means to open the file in write mode so I can create/overwrite stuff, and f is just asigning the opened file to a variable for json.dump()

</div>